<a href="https://colab.research.google.com/github/psh135230-design/BioAI_KU_2026/blob/main/1106.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/psh135230-design/BioAI_KU_2026/blob/main/%EB%B0%94%EC%9D%B4%EC%98%A4%EC%9D%B8%EA%B3%B5%EC%A7%80%EB%8A%A5_10_13(%ED%99%94).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -q "scanpy[leiden]==1.11.5" "pandas==2.2.3" "scipy==1.15.3" "scikit-image==0.25.2" tables seaborn


In [ ]:
from pathlib import Path
import json, hashlib, re, platform
from importlib.metadata import version
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import adjusted_rand_score
from IPython.display import display

SEED = 0
OUT = Path("mye_results")
OUT.mkdir(exist_ok=True)
sc.settings.figdir = OUT
sc.settings.autoshow = True
versions = {p: version(p) for p in ["scanpy", "anndata", "numpy", "pandas", "scipy", "scikit-learn", "leidenalg", "igraph"]}
print(versions)
(OUT / "versions.json").write_text(json.dumps(versions, indent=2))

def dense(x):
    return x.toarray() if sparse.issparse(x) else np.asarray(x)

def available(panel):
    return {k: keep for k, gs in panel.items() if (keep := [g for g in gs if g in adata.var_names])}

Path("figures_1101").mkdir(exist_ok=True)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)


In [ ]:
FILE = Path("/content/GSE154763_MYE_count.h5")
if not FILE.exists():
    local = Path("GSE154763_MYE_count.h5")
    if local.exists():
        FILE = local
    else:
        try:
            from google.colab import files
        except ImportError:
            raise FileNotFoundError("GSE154763_MYE_count.h5를 작업 폴더에 넣거나 FILE 경로를 수정하세요.")
        uploaded = files.upload()
        candidates = [Path(name) for name in uploaded if name.endswith(".h5")]
        if len(candidates) != 1:
            raise ValueError("MYE raw count h5 파일 한 개를 업로드하세요.")
        FILE = candidates[0]

raw_adata = sc.read_10x_h5(FILE)
raw_adata.var_names_make_unique()
raw_adata.X = sparse.csr_matrix(raw_adata.X)
v = raw_adata.X.data
if not np.isfinite(v).all() or (v < 0).any() or not np.allclose(v, np.round(v)):
    raise ValueError("입력 파일은 비음수 정수 raw count여야 합니다.")
if not raw_adata.obs_names.is_unique:
    raise ValueError("세포 바코드가 중복됩니다.")
sample = raw_adata.obs_names.to_series().str.extract(r"-(\d+)$", expand=False)
if sample.isna().any():
    raise ValueError("바코드 접미사가 없는 세포가 있습니다. 샘플 대응표가 필요합니다.")
raw_adata.obs["sample"] = sample.astype("category")
raw_adata.var["mt"] = np.asarray(raw_adata.var_names.str.startswith("MT-"), dtype=bool)
sc.pp.calculate_qc_metrics(raw_adata, qc_vars=["mt"], percent_top=None, inplace=True)
print(raw_adata)
display(raw_adata.obs["sample"].value_counts().sort_index())
raw_adata.obs.to_csv(OUT / "input_cell_qc.csv", index_label="cell_barcode")
input_hash = hashlib.sha256("\n".join(raw_adata.obs_names).encode()).hexdigest()

# 이후 원본 셀은 adata라는 이름을 사용합니다.
adata = raw_adata.copy()


In [ ]:
print("Data shape:", adata.shape)

print("\nCell metadata:")
print(adata.obs.columns.tolist())

print("\nGene metadata:")
print(adata.var.columns.tolist())

print("\nSaved embeddings:")
print(list(adata.obsm.keys()))

print("\nSaved analyses:")
print(list(adata.uns.keys()))

In [ ]:
adata.var_names_make_unique()

# astype(bool)을 추가하여 Pandas BooleanArray 대신 NumPy boolean 배열로 변환하여 저장합니다.
adata.var['mt'] = adata.var_names.str.startswith('MT-').astype(bool)

sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=['mt'],
    percent_top=None,
    inplace=True
)

print(adata.obs[
    ['n_genes_by_counts', 'total_counts', 'pct_counts_mt']
].describe())

In [ ]:
# 총 count vs 검출 유전자 수 (색 = 미토 비율)
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts', color='pct_counts_mt',
              title='Detected genes vs total counts')

# 총 count vs 미토 비율
sc.pl.scatter(adata, x='total_counts', y='pct_counts_mt',
              title='Mito % vs total counts')

In [ ]:
import numpy as np
from scipy import sparse

if sparse.issparse(adata.X):
    values = adata.X.data
else:
    values = np.asarray(adata.X).ravel()

print("Minimum:", values.min())
print("Maximum:", values.max())

# 모든 값이 정수인지 확인
print(
    "Integer counts:",
    np.allclose(values, np.round(values))
)

# 음수 존재 여부
print(
    "Non-negative:",
    np.all(values >= 0)
)

In [ ]:
# 정규화 재실행 시에도 원본 복사본에서 시작: 이중 정규화 방지
adata = raw_adata.copy()
adata.layers["counts"] = sparse.csr_matrix(raw_adata.X.copy())
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
print(adata)


In [ ]:
import numpy as np

# Raw count 데이터
raw_counts = adata.layers["counts"]

# 정규화 및 로그 변환된 데이터
normalized_data = adata.X

print("Raw counts shape:", raw_counts.shape)
print("Normalized data shape:", normalized_data.shape)

print("\nLog transformation 정보:")
print(adata.uns.get("log1p"))

print("\nRaw data:")
print(raw_counts[:5, :5].toarray())

print("\nNormalized & Log-transformed data:")
print(normalized_data[:5, :5].toarray())

In [ ]:
# HVG 2,000개 선정 (논문에서 STAR methods 분석할 때 HGV 2천 개 선정했다고 했음)
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,
    flavor='seurat'
)

# 선정된 HVG 개수 확인
print(
    "Number of HVGs:",
    adata.var['highly_variable'].sum()
)

# HVG 시각화
sc.pl.highly_variable_genes(adata)

## 3. ARI 비교와 실행 조건 기록
질문: 같은 입력 세포에서 PC 수만 30에서 50으로 바꿨을 때 군집화가 얼마나 달라지는가?
resolution 1.0과 0.6을 각각 별도 비교합니다. 1.0은 PDF 첫 실행의 기본값을 명시한 것이고, 0.6은 기존 최종 주석 설정을 재현하기 위한 것입니다. 이 값들이 최적이라는 뜻은 아닙니다.
PCA는 앞 셀에서 한 번만 계산합니다. 같은 전처리·HVG·PCA의 앞 30/50개를 사용하고, k=15와 seed=0 및 Leiden 구현을 고정합니다. ARI는 대응 라벨로 바꾸지 않고 원래 라벨로 계산합니다. 다른 seed에서의 안정성이나 생물학적 정확도를 검증하는 실험은 아닙니다.


In [ ]:
# PCA를 한 번만 계산하고 PC30/50 비교에서 공유
sc.tl.pca(adata, n_comps=50, mask_var="highly_variable", svd_solver="arpack", random_state=SEED)
# 한 입력 객체에서 그래프를 만들고 결과 라벨을 별도로 보존
if adata.obsm["X_pca"].shape[1] < 50:
    raise ValueError("앞 PCA 셀에서 50개 성분을 먼저 계산하세요.")
if not adata.obs_names.is_unique:
    raise ValueError("입력 바코드가 중복됩니다.")

comparison_index = adata.obs_names.copy()
for pc in [10, 20, 30, 50]:
    sc.pp.neighbors(adata, n_pcs=pc, n_neighbors=15, random_state=SEED,
                    key_added=f"nn{pc}")

# 배열 위치가 아니라 바코드로 연결. 누락/추가/중복은 조용히 제외하지 않고 중단.
def align_partitions(a, b):
    if not a.index.is_unique or not b.index.is_unique:
        raise ValueError("군집 결과에 중복 바코드가 있습니다.")
    if a.isna().any() or b.isna().any():
        raise ValueError("군집 라벨에 누락값이 있습니다.")
    if len(a.index.difference(b.index)) or len(b.index.difference(a.index)):
        raise ValueError("두 군집 결과의 세포 집합이 다릅니다.")
    b = b.reindex(a.index)
    assert a.index.equals(b.index)
    return a.astype(str), b.astype(str)

runs, partitions = [], {}
for resolution, tag, pcs in [(1.0, "res10", [10,20,30,50]),
                             (0.6, "res06", [30,50])]:
    for pc in pcs:
        key = f"leiden_pc{pc}_{tag}"
        sc.tl.leiden(adata, neighbors_key=f"nn{pc}", resolution=resolution,
                     random_state=SEED, flavor="leidenalg", n_iterations=-1,
                     directed=True, use_weights=True, key_added=key)
        labels = adata.obs[key].copy()
        if not labels.index.equals(comparison_index):
            raise ValueError("실행 중 세포 순서가 달라졌습니다.")
        partitions[key] = labels
        labels.to_csv(OUT / f"{key}_labels.csv", index_label="cell_barcode")
        runs.append({"run":key, "n_cells":adata.n_obs,
                     "cell_order_sha256":input_hash, "n_pcs":pc,
                     "n_neighbors":15, "resolution":resolution, "seed":SEED,
                     "leiden_flavor":"leidenalg", "directed":True,
                     "use_weights":True, "n_iterations":-1,
                     "n_clusters":labels.nunique(), "batch_correction":False,
                     "input":"same log1p matrix / HVG2000 / one PCA50",
                     "code_location":"section 3 ARI comparison cell"})

run_table = pd.DataFrame(runs)
run_table.to_csv(OUT / "clustering_run_conditions.csv", index=False)
ari_rows = []
for tag in ["res10", "res06"]:
    ka, kb = f"leiden_pc30_{tag}", f"leiden_pc50_{tag}"
    a, b = align_partitions(partitions[ka], partitions[kb])
    value = adjusted_rand_score(a, b)
    ari_rows.append({"comparison":tag, "label_a":ka, "label_b":kb,
                     "n_cells":len(a), "ARI":value,
                     "clusters_pc30":a.nunique(), "clusters_pc50":b.nunique(),
                     "cell_set_match":True, "order_aligned_by_barcode":True})
ari_table = pd.DataFrame(ari_rows)
ari_table.to_csv(OUT / "ari_comparisons.csv", index=False)
print(run_table.drop(columns=["cell_order_sha256"]).to_string(index=False))
print(ari_table.to_string(index=False))
# 0.82 / 0.834는 과거 출력입니다. 새 값을 강제하지 않습니다.
adata.obs["leiden_30"] = adata.obs["leiden_pc30_res06"].copy()
adata.obs["leiden_50"] = adata.obs["leiden_pc50_res06"].copy()
sc.tl.umap(adata, neighbors_key="nn30", random_state=SEED)


## 4. 667개 산출 규칙과 분할·병합 점검
기존 규칙 보존: PC30의 각 군집에서 가장 많은 세포가 이동한 PC50 군집을 대표 대응으로 정합니다. 그 대표 대응 밖의 세포 수를 셉니다. 동률이면 숫자가 작은 라벨을 선택하고 동률 여부를 표에 남깁니다. 동률 시 대응 선택은 임의적이라는 한계가 있습니다.
이 규칙은 PC30 기준의 방향성 있는 요약이며 일대일 대응이 아닙니다. 순수 병합은 변경 0개로 계산될 수 있습니다. 따라서 667개를 전체 구조 변화에 관여한 세포 수로 해석하지 않습니다.
역방향 최빈 대응과 각 행/열의 전체 구성도 함께 저장합니다. 역방향 수치를 새로운 정답으로 사용하거나 두 방향을 합쳐 전체 변경 수로 부르지 않습니다. ARI는 이 대응 규칙과 독립적으로 계산됩니다.


In [ ]:
def numeric_order(values):
    return sorted(set(map(str, values)), key=lambda s: int(s))

# 계산 규칙을 함수로 분리하여 실제 데이터와 작은 반례에 동일하게 적용
def modal_correspondence(a, b):
    a, b = align_partitions(a, b)
    table = pd.crosstab(a.rename("pc30"), b.rename("pc50"))
    table = table.reindex(index=numeric_order(a), columns=numeric_order(b), fill_value=0)
    forward = table.idxmax(axis=1)  # 숫자 오름차순으로 정렬 후 동률 첫 값
    reverse = table.idxmax(axis=0)
    frame = pd.DataFrame({"pc30":a, "pc50":b})
    frame["pc50_modal_from_pc30"] = frame["pc30"].map(forward)
    frame["pc30_modal_from_pc50"] = frame["pc50"].map(reverse)
    frame["off_pc30_modal"] = frame["pc50"].ne(frame["pc50_modal_from_pc30"])
    frame["off_pc50_modal"] = frame["pc30"].ne(frame["pc30_modal_from_pc50"])
    row = pd.DataFrame({"pc50_modal_label":forward, "n_cells":table.sum(axis=1),
                        "n_modal":table.max(axis=1),
                        "n_tied_maxima":table.eq(table.max(axis=1), axis=0).sum(axis=1),
                        "n_pc50_destinations":table.gt(0).sum(axis=1)})
    row["n_off_modal"] = row["n_cells"] - row["n_modal"]
    row["off_modal_percent"] = row["n_off_modal"] / row["n_cells"] * 100
    column = pd.DataFrame({"pc30_modal_label":reverse, "n_cells":table.sum(axis=0),
                           "n_modal":table.max(axis=0),
                           "n_tied_maxima":table.eq(table.max(axis=0), axis=1).sum(axis=0),
                           "n_pc30_sources":table.gt(0).sum(axis=0)})
    column["n_off_modal"] = column["n_cells"] - column["n_modal"]
    return table, frame, row, column

# 의미가 알려진 작은 사례: 번호 치환, 병합, 분할, 순서 재정렬
idx = pd.Index(["cell_a","cell_b","cell_c","cell_d"])
a_test = pd.Series(["0","0","1","1"], index=idx)
renamed = pd.Series(["8","8","9","9"], index=idx)
assert adjusted_rand_score(*align_partitions(a_test, renamed.iloc[::-1])) == 1
assert modal_correspondence(a_test, renamed)[1]["off_pc30_modal"].sum() == 0
merged = pd.Series(["0","0","0","0"], index=idx)
assert modal_correspondence(a_test, merged)[1]["off_pc30_modal"].sum() == 0
assert adjusted_rand_score(a_test, merged) < 1
split = pd.Series(["0","1","2","2"], index=idx)
assert modal_correspondence(a_test, split)[1]["off_pc30_modal"].sum() == 1
print("규칙 확인 통과: 순서 재정렬 / 라벨 치환 / 병합의 사각지대 / 분할")

ct, cell_map, mapping, reverse_mapping = modal_correspondence(
    adata.obs["leiden_30"], adata.obs["leiden_50"])
cell_map.insert(0, "sample", adata.obs["sample"].reindex(cell_map.index))
changed = cell_map["off_pc30_modal"].reindex(adata.obs_names)
adata.obs["changed"] = pd.Categorical(np.where(changed,"changed","same"), categories=["changed","same"])
adata.uns["changed_colors"] = ["red","lightgrey"]
order30, order50 = ct.index.tolist(), ct.columns.tolist()
ct.to_csv(OUT / "pc30_pc50_full_contingency.csv")
ct.div(ct.sum(axis=1),axis=0).to_csv(OUT / "pc30_pc50_row_fractions.csv")
ct.div(ct.sum(axis=0),axis=1).to_csv(OUT / "pc30_pc50_column_fractions.csv")
mapping.to_csv(OUT / "cluster_mapping_rule.csv")
reverse_mapping.to_csv(OUT / "reverse_cluster_mapping.csv")
cell_map.to_csv(OUT / "cell_correspondence.csv", index_label="cell_barcode")
counts = pd.DataFrame([{"direction":"PC30 to PC50", "n_off_modal":int(changed.sum()),
                        "percent":float(changed.mean()*100)},
                       {"direction":"PC50 to PC30", "n_off_modal":int(cell_map["off_pc50_modal"].sum()),
                        "percent":float(cell_map["off_pc50_modal"].mean()*100)}])
counts.to_csv(OUT / "directional_off_modal_counts.csv",index=False)
print(counts.to_string(index=False))
print(mapping.to_string())
print(reverse_mapping.to_string())
print("과거 667개와의 일치는 실행 결과로 확인합니다. 값을 강제하지 않습니다.")
sc.pl.umap(adata,color=["leiden_30","changed"],save="_changed_res06.png")
plt.figure(figsize=(12,7))
sns.heatmap(ct.div(ct.sum(axis=1),axis=0)*100,cmap="Reds")
plt.tight_layout(); plt.savefig(OUT / "correspondence_heatmap.png",dpi=200); plt.show()
change_rate=adata.obs.assign(is_changed=changed).groupby("sample",observed=True)["is_changed"].mean()*100
change_rate.to_csv(OUT / "changed_percent_by_sample.csv")
display(change_rate)

# 5. 세포형 추정 (Cheng 2021 Fig 1E 마커)
markers = {
    'Mono_CD14':  ['FCN1', 'S100A9', 'S100A8'],
    'Mono_CD16':  ['FCGR3A', 'LST1', 'LILRB2'],
    'Macro_C1QC': ['C1QC', 'C1QA', 'APOE'],
    'Macro_NLRP3':['NLRP3', 'EREG', 'IL1B'],
    'cDC2':       ['CD1C', 'FCER1A', 'HLA-DQA1'],
    'cDC3':       ['LAMP3', 'CCR7', 'FSCN1'],
    'pDC':        ['LILRA4', 'GZMB', 'IL3RA'],
    'T_cell':     ['CD3E', 'CD3D', 'TRAC'],
}
markers = {k: [g for g in v if g in adata.var_names] for k, v in markers.items()}
markers = {k: g for k, g in markers.items() if g}
for k, g in markers.items():
    sc.tl.score_genes(adata, g, score_name=f'score_{k}', random_state=SEED, use_raw=False)
adata.obs['pred_type'] = adata.obs[[f'score_{k}' for k in markers]].idxmax(axis=1).str.replace('score_', '')
print((pd.crosstab(adata.obs['pred_type'], adata.obs['changed'], normalize='columns') * 100).round(1))
sc.pl.dotplot(adata, markers, groupby='leiden_30', save='_celltype_res06.png')

# 6. 글로 요약
obs = adata.obs
n_ch = (obs['changed'] == 'changed').sum()
print(f"\n전체 {len(obs)}개 세포 중 {n_ch}개({n_ch/len(obs)*100:.1f}%)가 PC30 기준 최빈 대응 밖에 있음.")
for cl in sorted(obs['leiden_30'].unique(), key=int):
    sub = obs[obs['leiden_30'] == cl]
    chs = sub[sub['changed'] == 'changed']
    if len(chs) < 10:
        continue
    types = ', '.join(f"{k} {v:.0f}%" for k, v in (chs['pred_type'].value_counts(normalize=True) * 100).head(3).items())
    print(f"군집 {cl}: 주 세포형은 {sub['pred_type'].value_counts().idxmax()}임. "
          f"{len(sub)}개 중 {len(chs)}개({len(chs)/len(sub)*100:.0f}%)가 바뀌었고, 주로 PC50 군집 "
          f"{chs['leiden_50'].value_counts().idxmax()}로 옮겼음. 바뀐 세포는 {types}이고, "
          f"가장 많은 샘플은 {chs['sample'].value_counts().idxmax()}임.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

obs = adata.obs
order30 = sorted(obs['leiden_30'].unique(), key=int)
fig, ax = plt.subplots(2, 3, figsize=(18, 11))

# A. 교차표 히트맵 (행 = PC30 군집, 각 행을 100%로)
ct = pd.crosstab(obs['leiden_30'], obs['leiden_50'], normalize='index') * 100
sns.heatmap(ct.loc[order30], cmap='Reds', vmin=0, vmax=100, ax=ax[0, 0], cbar_kws={'label': '% of PC30 cluster'})
ax[0, 0].set_title('A. PC30 → PC50 cluster correspondence (res 0.6)')

# B. UMAP: 바뀐 세포
sc.pl.umap(adata, color='changed', palette=['red', 'lightgrey'], ax=ax[0, 1], show=False,
           title=f"B. Changed cells ({(obs['changed']=='changed').sum()}, {(obs['changed']=='changed').mean()*100:.1f}%)")

# C. 군집별 변경 비율
rate = obs.groupby('leiden_30', observed=True)['changed'].apply(lambda x: (x == 'changed').mean() * 100).loc[order30]
ax[0, 2].bar(rate.index.astype(str), rate.values, color=['red' if v >= 10 else 'grey' for v in rate.values])
ax[0, 2].axhline(10, ls='--', c='k', lw=0.8)
ax[0, 2].set(title='C. % changed per PC30 cluster (red ≥10%)', xlabel='PC30 cluster', ylabel='% changed')

# D. 세포형 구성: 바뀐 vs 그대로
comp = pd.crosstab(obs['changed'], obs['pred_type'], normalize='index') * 100
comp.plot(kind='barh', stacked=True, ax=ax[1, 0], colormap='tab10', legend=False)
ax[1, 0].set(title='D. Predicted cell type (marker score)', xlabel='%')
ax[1, 0].legend(bbox_to_anchor=(1, 1), fontsize=8)

# E. 샘플별 변경 비율
srate = obs.groupby('sample', observed=True)['changed'].apply(lambda x: (x == 'changed').mean() * 100)
ax[1, 1].bar(srate.index.astype(str), srate.values, color='steelblue')
ax[1, 1].set(title='E. % changed per sample', xlabel='sample', ylabel='% changed')

# F. QC 비교
sns.violinplot(data=obs, x='changed', y='n_genes_by_counts', ax=ax[1, 2], cut=0)
ax[1, 2].set(title='F. Genes per cell (QC comparison)')

plt.tight_layout()
plt.savefig('figures_1101/F_changed_summary_res06.png', dpi=200, bbox_inches='tight')
plt.show()

# ===== 자동 해설 =====
n_ch = (obs['changed'] == 'changed').sum()
hi = rate[rate >= 10]
c_comp = comp.reindex(["changed", "same"], fill_value=0).loc['changed'].sort_values(ascending=False)
s_comp = comp.reindex(['changed', 'same'], fill_value=0).loc['same']
med = obs.groupby('changed', observed=True)['n_genes_by_counts'].median()
print("[A] 행별 집중도는 PC30 군집의 주 대응을 보여줌. 병합은 열별 구성도 함께 확인해야 함.")
print(f"[B] 전체 {len(obs)}개 중 {n_ch}개({n_ch/len(obs)*100:.1f}%)가 바뀌었음. 빨간 점은 PC30 기준 최빈 대응 밖의 세포이며 생물학적 경계를 확정하지 않음.")
print(f"[C] 변경 비율 10% 이상 군집: " + ', '.join(f"{k}({v:.0f}%)" for k, v in hi.sort_values(ascending=False).items()) + ".")
print("[D] 바뀐 세포 세포형: " + ', '.join(f"{k} {v:.1f}%(그대로 {s_comp[k]:.1f}%)" for k, v in c_comp.head(4).items()) + ".")
print(f"[E] 샘플별 변경 비율은 {srate.min():.1f}~{srate.max():.1f}%이고, 가장 높은 샘플은 {srate.idxmax()}임.")
print(f"[F] 유전자 수 중앙값은 바뀐 세포 {med['changed']:.0f}, 그대로 {med['same']:.0f}임. 이 중앙값 비교만으로 doublet을 배제할 수 없음.")
print("※ 세포형은 마커 점수로 추정한 값이라 확정 주석이 아님.")

In [ ]:
# 앞서 계산한 PCA50을 유지하고 첫 30개 성분의 설명력을 확인합니다.
# PCA 결과 확인
print("PCA shape:", adata.obsm['X_pca'].shape)


# 설명 분산 비율 시각화
sc.pl.pca_variance_ratio(
    adata,
    n_pcs=30,
    log=False
)

variance_ratio = adata.uns["pca"]["variance_ratio"][:5]
pcs = [f"PC{i+1}" for i in range(len(variance_ratio))]
explained_pct = variance_ratio * 100
cumulative_pct = np.cumsum(explained_pct)

pca_summary_df = pd.DataFrame(
    {
        "주성분 (PC)": pcs,
        "설명 분산 비율 (%)": np.round(explained_pct, 2),
        "누적 설명 분산 비율 (%)": np.round(cumulative_pct, 2),
    }
)

# 표 출력
pd.set_option("display.max_rows", 30)
display(pca_summary_df)

In [ ]:
import numpy as np, pandas as pd, scanpy as sc, matplotlib.pyplot as plt

# 'leiden' 대신 정의된 'leiden_30' 컬럼의 고유 클러스터 개수를 출력합니다.
if 'leiden_30' in adata.obs:
    print(f"leiden_30 클러스터 개수: {adata.obs['leiden_30'].nunique()}개 (기존 실행은 14개; 새 결과를 강제하지 않음)")
elif 'leiden' in adata.obs:
    print(f"leiden 클러스터 개수: {adata.obs['leiden'].nunique()}개")
else:
    print("경고: 'leiden_30' 또는 'leiden' 클러스터 정보가 adata.obs에 없습니다.")

print("'counts' 레이어 보존 여부:", 'counts' in adata.layers)
# 'counts'가 없는(False) 경우: 노트북 맨 위 normalize_total 직전에 adata.layers['counts'] = adata.X.copy() 넣고 다시 실행

In [ ]:
markers = {
    'Mono':      ['CD14', 'FCN1', 'S100A8', 'VCAN'],
    'CD16 Mono': ['FCGR3A'],
    'Macro':     ['CD68', 'CD163', 'C1QA', 'C1QC', 'APOE', 'MRC1', 'MAF'],
    'MHC-II':    ['HLA-DRA', 'HLA-DQA1', 'CD74'],
    'cDC2':      ['CD1C', 'FCER1A', 'CLEC10A', 'CD1E'],
    'cDC1':      ['CLEC9A', 'XCR1'],
    'mregDC':    ['LAMP3', 'CCR7'],
    'moDC':      ['CD209'],
    'Inflam':    ['NLRP3', 'IL1B', 'EREG'],
    'T/NK':      ['CD3E', 'NKG7'],
    'B/Plasma':  ['CD79A', 'JCHAIN', 'MZB1', 'IGKC'],
    'Platelet/RBC': ['PPBP', 'HBB'],
}
# groupby 컬럼을 기존 'leiden'에서 실제 존재하는 'leiden_30'으로 변경합니다.
plot_markers = available(markers)
sc.pl.dotplot(adata, plot_markers, groupby='leiden_30', standard_scale='var', cmap='Reds',
              save='_F3_ambiguous_multimarker.png')

# 군집 3, 6, 11의 발현 세포 비율(%) 표로 확인 (leiden -> leiden_30)
genes = ['C1QC','APOE','MRC1','CD163','CD1C','FCER1A','CLEC10A','CLEC9A','JCHAIN','MZB1']
genes = [g for g in genes if g in adata.var_names]
sub = adata[adata.obs['leiden_30'].isin(['3','6','11']), genes]
pct = pd.DataFrame((sub.X > 0).toarray() if hasattr(sub.X,'toarray') else (sub.X > 0),
                   columns=genes, index=sub.obs_names).groupby(sub.obs['leiden_30'].values).mean()*100
print(pct.round(0))

## 7. 다중 마커와 군집 11의 같은 세포 내 발현
raw count > 0을 검출 기준으로 사용합니다. 동시 검출은 기능적 정체성이나 doublet 여부를 단독으로 확정하지 않습니다. 군집 번호가 기존과 달라지면 마커를 보고 TARGET_CLUSTER를 변경하세요.


In [ ]:
TARGET_CLUSTER="11"
co_genes=["CD1C","FCER1A","CLEC9A","XCR1"]
missing=[x for x in co_genes if x not in adata.var_names]
print("미검출 유전자(행렬에 없음):",missing)
mask=adata.obs["leiden_30"].astype(str).eq(TARGET_CLUSTER)
if mask.any() and all(x in adata.var_names for x in ["CD1C","CLEC9A"]):
    sub=adata[mask]
    genes=[x for x in co_genes if x in adata.var_names]
    per_cell=pd.DataFrame(dense(sub[:,genes].layers["counts"]),index=sub.obs_names,columns=genes)
    per_cell["sample"]=sub.obs["sample"]
    a,b=per_cell["CD1C"]>0,per_cell["CLEC9A"]>0
    per_cell["detection_group"]=np.select([a&b,a&~b,~a&b],["both","CD1C_only","CLEC9A_only"],default="neither")
    per_cell.to_csv(OUT / "cluster11_coexpression_by_cell.csv",index_label="cell_barcode")
    tab=per_cell["detection_group"].value_counts().reindex(["both","CD1C_only","CLEC9A_only","neither"],fill_value=0).to_frame("n_cells")
    tab["percent"]=tab["n_cells"]/len(per_cell)*100
    tab.to_csv(OUT / "cluster11_coexpression_summary.csv"); display(tab)
    display(pd.crosstab(per_cell["sample"],per_cell["detection_group"]))
else:
    print("대상 군집 또는 필수 유전자가 없어 동시 발현 계산을 생략합니다.")


## 5. 샘플별 Scrublet
raw count 전체 유전자를 사용합니다. 추정 샘플별로 점수·자동 임계값을 계산하며 임계값을 강제로 조정하지 않습니다. 같은 계열 doublet이나 RNA 혼입을 이 결과만으로 배제하지 않습니다.


In [ ]:
# csr_array의 메서드를 전역 패치하지 않고 입력을 csr_matrix로 명시합니다.
scrub = sc.AnnData(X=sparse.csr_matrix(raw_adata.X.copy()),
                   obs=raw_adata.obs[["sample"]].copy(),var=raw_adata.var.copy())
sc.pp.scrublet(scrub,batch_key="sample",random_state=SEED,expected_doublet_rate=0.05,
               sim_doublet_ratio=2.0)
for col in ["doublet_score","predicted_doublet"]:
    adata.obs[col]=scrub.obs[col].reindex(adata.obs_names)
if adata.obs[["doublet_score","predicted_doublet"]].isna().any().any():
    raise ValueError("Scrublet 결과에 누락값이 있습니다.")
summary=scrub.obs.groupby("sample",observed=True).agg(
    n_cells=("doublet_score","size"),n_predicted=("predicted_doublet","sum"),
    predicted_fraction=("predicted_doublet","mean"),max_score=("doublet_score","max"))
batches=scrub.uns["scrublet"]["batches"]
summary["threshold"]=[batches[str(s)].get("threshold",np.nan) for s in summary.index]
summary["predicted_percent"]=summary["predicted_fraction"]*100
summary.to_csv(OUT / "scrublet_by_sample.csv")
scrub.obs.to_csv(OUT / "scrublet_by_cell.csv",index_label="cell_barcode")
print("전체 예측 doublet:",int(adata.obs["predicted_doublet"].sum()),"/",adata.n_obs)
print("전체 예측 비율(%):",adata.obs["predicted_doublet"].mean()*100)
print(summary.to_string())
# show=False로 Figure가 닫히기 전에 저장 (이전 저장 방식은 빈 그림 가능)
score_fig = sc.pl.scrublet_score_distribution(scrub, return_fig=True, show=False)
score_fig.savefig(OUT / "scrublet_score_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

ig=np.asarray(adata.var_names.str.match(r"^IG[HKL][VCJ]"),dtype=bool)
adata.obs["pct_ig"]=np.asarray(adata.layers["counts"][:,ig].sum(axis=1)).ravel()/adata.obs["total_counts"].to_numpy()*100
g=adata.obs.groupby("leiden_30",observed=True)
qc=g[["total_counts","n_genes_by_counts","pct_counts_mt","doublet_score"]].median()
qc["predicted_doublet_percent"]=g["predicted_doublet"].mean()*100
qc["Ig_above_1_percent_cells"]=g["pct_ig"].apply(lambda s:(s>1).mean()*100)
qc.to_csv(OUT / "qc_by_cluster.csv"); display(qc)
sc.pl.violin(adata,["total_counts","n_genes_by_counts","pct_counts_mt","doublet_score","pct_ig"],
             groupby="leiden_30",rotation=90,stripplot=False,save="_qc_doublet.png")
composition=pd.crosstab(adata.obs["leiden_30"],adata.obs["sample"])
composition.to_csv(OUT / "cluster_by_sample.csv"); display(composition)
composition.div(composition.sum(axis=1),axis=0).plot.bar(stacked=True,figsize=(10,4))
plt.tight_layout(); plt.savefig(OUT / "cluster_sample.png",dpi=200); plt.show()
for gene in ["XIST","DDX3Y","RPS4Y1"]:
    if gene in adata.var_names:
        frame=adata.obs[["leiden_30"]].copy()
        frame["expressed"]=dense(adata[:,gene].X).ravel()>0
        print(gene,frame.groupby("leiden_30",observed=True)["expressed"].mean().to_dict())


In [ ]:
import re
stress = ['FOS','FOSB','JUN','JUNB','EGR1','DUSP1','IER2','IER3','HSPA1A','HSPA1B','HSP90AA1']
hvg = adata.var_names[adata.var['highly_variable']]
print('HVG 안 스트레스 유전자:', [g for g in stress if g in hvg])
print('HVG 안 Ig 유전자 수:', hvg.str.match(r'^IG[HKL][VCJ]').sum())
print('HVG 안 리보솜 유전자 수:', hvg.str.match(r'^RP[LS]').sum())

In [ ]:
import re
filter_v1 = re.compile(r'^(RPL|RPS|MT-|IG[HKL][VCJ]|FOS|JUN|DUSP1|IER|HSPA|HSPH|HSP90)')
filter_v2 = re.compile(r'^(RPL|RPS|MT-|IG[HKL][VCJ]|FOS|JUN|DUSP1|IER|EGR1|HSPA|HSPH|HSP90)')

sc.tl.rank_genes_groups(adata, 'leiden_30', method='wilcoxon', key_added='wx', use_raw=False)

rows = []
for c in adata.obs['leiden_30'].cat.categories:
    df = sc.get.rank_genes_groups_df(adata, c, key='wx')
    v1 = [g for g in df['names'] if not filter_v1.match(g)][:10]
    v2 = [g for g in df['names'] if not filter_v2.match(g)][:10]
    removed = [g for g in v1 if g not in v2]
    rows.append({
        'Cluster': c,
        'n_cells': int((adata.obs['leiden_30'] == c).sum()),
        'Top 10 markers (revised filter)': ', '.join(v2),
        'Removed by revision': ', '.join(removed) if removed else '–',
        'Newly included': ', '.join([g for g in v2 if g not in v1]) or '–',
    })
tbl = pd.DataFrame(rows).set_index('Cluster')
tbl.to_csv('figures_1101/T6_top_markers_revised_filter.csv')
pd.set_option('display.max_colwidth', None)
tbl

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden_30', method='t-test',   key_added='tt', use_raw=False)
sc.tl.rank_genes_groups(adata, 'leiden_30', method='wilcoxon', key_added='wx', use_raw=False)

def top10(key, c):
    names = sc.get.rank_genes_groups_df(adata, c, key=key)['names']
    return [g for g in names if not filter_v2.match(g)][:10]

rows = []
for c in adata.obs['leiden_30'].cat.categories:
    t, w = top10('tt', c), top10('wx', c)
    rows.append({'Cluster': c,
                 'Overlap (of 10)': len(set(t) & set(w)),
                 't-test top 10': ', '.join(t),
                 'Wilcoxon top 10': ', '.join(w),
                 't-test only': ', '.join([g for g in t if g not in w]) or '–',
                 'Wilcoxon only': ', '.join([g for g in w if g not in t]) or '–'})
de = pd.DataFrame(rows).set_index('Cluster')
de.to_csv('figures_1101/T4_DE_ttest_vs_wilcoxon.csv')
pd.set_option('display.max_colwidth', None)
print('평균 겹침:', round(de['Overlap (of 10)'].mean(), 1))
de[['Overlap (of 10)', 't-test only', 'Wilcoxon only']]

In [ ]:
# 3 vs 6: 완전한 유전자 순위를 보존하고, 통계량의 방향으로 상위 유전자 선정
if {"3","6"}.issubset(set(adata.obs["leiden_30"].astype(str))):
    for method in ["t-test","wilcoxon"]:
        key=f"3v6_{method}"
        sc.tl.rank_genes_groups(adata,"leiden_30",groups=["3"],reference="6",method=method,
                               key_added=key,use_raw=False,n_genes=adata.n_vars)
        frame=sc.get.rank_genes_groups_df(adata,"3",key=key)
        frame.to_csv(OUT / f"cluster3_vs6_{method}_all.csv",index=False)
        clean=frame[~frame["names"].str.match(filter_v2.pattern)]
        display(clean[clean["scores"]>0].nlargest(10,"scores"))
        display(clean[clean["scores"]<0].nsmallest(10,"scores"))


In [ ]:
# ARI에서 사용한 최종 PC30/k15/res0.6 라벨을 후속 셀에 그대로 연결
# 후속 분석에서 군집을 새로 만들거나 기존 라벨을 덮어쓰지 않습니다.
adata.obs["leiden"] = adata.obs["leiden_30"].copy()
sc.tl.umap(adata, neighbors_key="nn30", random_state=SEED)


In [ ]:
sensitivity=[]
for k in [10,15,30]:
    work=adata.copy()
    sc.pp.neighbors(work,n_pcs=30,n_neighbors=k,random_state=SEED)
    sc.tl.leiden(work,resolution=0.6,random_state=SEED,flavor="leidenalg",n_iterations=-1,directed=True,key_added="leiden_k")
    sc.tl.umap(work,random_state=SEED)
    sensitivity.append({"k":k,"n_clusters":work.obs["leiden_k"].nunique(),
                        "ARI_vs_final_k15":adjusted_rand_score(adata.obs["leiden_30"],work.obs["leiden_k"])})
    sc.pl.umap(work,color="leiden_k",title=f"k={k}, PC30, resolution=0.6",save=f"_k{k}.png")
pd.DataFrame(sensitivity).to_csv(OUT / "neighbor_sensitivity.csv",index=False)

print(pd.DataFrame(sensitivity).to_string(index=False))


In [ ]:
import scanpy as sc
import pandas as pd

# 각 클러스터의 Marker Gene 분석
sc.tl.rank_genes_groups(
    adata,
    groupby='leiden',
    method='t-test',
    use_raw=False,
    pts=True
)

print("Marker Gene Analysis is done")

In [ ]:
# 각 클러스터별 상위 7개 Marker Gene 추출

results = []

for cluster in adata.obs['leiden'].cat.categories:

    df = sc.get.rank_genes_groups_df(
        adata,
        group=cluster
    )

    # 유의한 Marker Gene 선정
    df = df[
        (df['pvals_adj'] < 0.05) &
        (df['logfoldchanges'] > 0.25)
    ]

    top_genes = df['names'].head(7).tolist()

    results.append({
        'Cluster': cluster,
        'Top Marker Genes': ', '.join(top_genes)
    })

marker_summary = pd.DataFrame(results)

display(marker_summary)

In [ ]:
import re
bad = re.compile(r'^(RPL|RPS|MT-|IGLV|IGKV|IGHV|IGLC|IGKC|FOS|JUN|DUSP1|IER)')
for c in adata.obs['leiden'].cat.categories:
    g = sc.get.rank_genes_groups_df(adata, group=c)['names']
    print(c, [x for x in g if not bad.match(x)][:7])

In [ ]:
import scanpy as sc

# 세포 유형별 대표 Marker gene
marker_genes = {
    "Mono_CD14":   ["FCN1", "S100A9", "S100A8"],    # Fig 1E
    "Mono_CD16":   ["FCGR3A", "LST1", "LILRB2"],    # Fig 1E
    "Mono/Macro":  ["CD68", "CD163"],               # Fig 1D
    "Macro_C1QC":  ["C1QC", "C1QA", "APOE"],        # Fig 1E
    "Macro_NLRP3": ["NLRP3", "EREG", "IL1B"],       # Fig 1E
    "cDC1":        ["CLEC9A", "FLT3", "IDO1"],      # Fig 1E
    "cDC2":        ["CD1C", "FCER1A", "HLA-DQA1"],  # Fig 1E, Fig 1D
    "cDC3_LAMP3":  ["LAMP3", "CCR7", "FSCN1"],      # Fig 1E
    "pDC":         ["LILRA4", "GZMB", "IL3RA"],     # Fig 1E, Fig 1D
    "Mast":        ["KIT", "TPSAB1", "CPA3"],       # Fig 1E, Fig 1D
}

tam_subsets = {   # 논문 TAM 아형
    "Macro_SPP1":  ["SPP1"],
    "Macro_C1QC":  ["C1QC"],
    "Macro_ISG15": ["ISG15"],
}

# 1. 데이터셋에 존재하는 유전자만 필터링하여 새로운 딕셔너리 생성
available_markers = {}
for cell_type, genes in marker_genes.items():
    available = [gene for gene in genes if gene in adata.var_names]
    if available:
        available_markers[cell_type] = available
        print(f"{cell_type} available markers: {available}")

# 2. 필터링된 딕셔너리를 통째로 전달하여 하나의 통합된 dotplot 생성
sc.pl.dotplot(
    adata,
    var_names=available_markers,
    groupby="leiden",
    standard_scale="var",
    use_raw=False,
    cmap="Reds" # 이미지와 비슷한 파랑-노랑-빨강 색상 맵 적용
)

In [ ]:
mast = ['KIT', 'TPSAB1', 'TPSB2', 'CPA3', 'MS4A2', 'HDC']
print('데이터에 있는 마스트 유전자:', [g for g in mast if g in adata.var_names])
if 'KIT' in adata.var_names:
    print('KIT 발현 세포 수:', int((adata[:, 'KIT'].X > 0).sum()))
else:
    print('KIT는 입력 행렬에 없습니다.')

### 기존 군집 번호 기반 주석표
아래 주석은 기존 해석을 유지한 후보 표입니다. 새 실행의 마커를 확인한 뒤 확정하세요. 추정 샘플을 환자로 간주하지 않으며, DC 세부 유형 및 대식세포 상태를 군집 번호만으로 확정하지 않습니다.

In [ ]:
import pandas as pd

annot_table = pd.DataFrame([
    ['2, 4, 5, 7, 10', 'Mono_CD14',                     'FCN1, S100A9, S100A8'],
    ['0, 8, 9',        'Mono_CD16',                     'FCGR3A, LST1, LILRB2'],
    ['1',              'Mono_CD14CD16 (중간형)',         'CD14·CD16 마커 둘 다 중간'],
    ['6',              'Macro_C1QC',                    'C1QC, C1QA, CD163'],
    ['3',              'Macro_C1QC (APOE 높은 쪽)',      'APOE, CD68, CD163, HLA-DQA1'],
    ['12',             'Macro_NLRP3',                   'NLRP3, EREG, IL1B'],
    ['11',             'cDC 후보 (cDC2/cDC1 다중 마커 검토 필요)',     'CD1C, FCER1A, HLA-DQA1'],
    ['13',             'pDC',                           'LILRA4, GZMB, IL3RA'],
    ['-',              'Mast (없음)',                    'KIT 거의 0'],
], columns=['클러스터', '세포 유형', '가장 진한 마커'])

# 클러스터별 세포 수 합치기
counts = adata.obs['leiden'].value_counts()
annot_table['세포 수'] = annot_table['클러스터'].apply(
    lambda s: sum(counts.get(c.strip(), 0) for c in s.split(',')))

annot_table

## 실행 조건과 결과 저장

In [ ]:
adata.obs.to_csv(OUT / "final_cell_metadata.csv",index_label="cell_barcode")
adata.var.to_csv(OUT / "gene_metadata.csv")
adata.write_h5ad(OUT / "MYE_analysis.h5ad",compression="gzip")
manifest={"input_file":FILE.name,"n_cells":adata.n_obs,"n_genes":adata.n_vars,
          "cell_order_sha256":input_hash,"versions":versions,"seed":SEED,
          "sample_source":"barcode_suffix_unverified","patient_metadata_available":False,
          "normalization_target":10000,"hvg_flavor":"seurat","hvg_n":2000,
          "doublets_removed":False,"additional_QC_filter":False,"batch_correction":False,
          "mapping_rule":"PC30 row modal PC50; ties choose smallest numeric PC50 label"}
(OUT / "run_manifest.json").write_text(json.dumps(manifest,indent=2,ensure_ascii=False))
print("완료:",OUT.resolve())
import shutil
archive=shutil.make_archive("MYE_results","zip",OUT)
print("결과 압축 파일:",archive)
